In [ ]:
# 1. Uninstall conflicting packages
!pip uninstall -y unsloth unsloth-zoo transformers tokenizers

# 2. Install unsloth with all dependencies (this will get the right versions)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 3. Restart runtime
# Go to: Runtime → Restart runtime (or Ctrl+M)

Found existing installation: unsloth 2026.1.3
Uninstalling unsloth-2026.1.3:
  Successfully uninstalled unsloth-2026.1.3
Found existing installation: unsloth_zoo 2026.1.3
Uninstalling unsloth_zoo-2026.1.3:
  Successfully uninstalled unsloth_zoo-2026.1.3
Found existing installation: transformers 4.57.3
Uninstalling transformers-4.57.3:
  Successfully uninstalled transformers-4.57.3
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-urvonbed/unsloth_57c86ea1015a4f24a6703fb49ade4f27
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-urvonbed/unsloth_57c86ea1015a4f24a6703fb49ade4f27
  Resolved https://github.com/unslothai/unsloth.git to commit ca0ecf1a3a404737f0de77f1fbec2e3bdf1c9d4e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metad

In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import pandas as pd
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
import pandas as pd
from datasets import Dataset

df = pd.read_json("/content/synthetic_training_data.jsonl",lines=True)
print(f" Loaded {len(df)} samples")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst sample:")
print(df.iloc[0])



 Loaded 1364 samples
Columns: ['Review', 'output']

First sample:
Review     crawls up the butt. hard to sleep in\n\nCute ...
output     Profanity: yes\nSentiment: negative\nRewritten: 
Name: 0, dtype: object


In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import pandas as pd
import torch

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Load model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="LiquidAI/LFM2.5-1.2B-Instruct",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

print(" Model loaded successfully!")

# Continue with rest of training code...

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
==((====))==  Unsloth 2026.1.3: Fast Lfm2 patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
 Model loaded successfully!


In [ ]:
# ============================================================================
# STEP 3: Apply LoRA
# ============================================================================

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

model.print_trainable_parameters()

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Unsloth: Making `model.base_model.model.model` require gradients
trainable params: 884,736 || all params: 1,171,225,344 || trainable%: 0.0755


In [ ]:
def format_lfm25_final(row):
    # ---------------------------------------------------------
    # 1. DEFINE THE INSTRUCTION
    # ---------------------------------------------------------
    # We explicitly ask ONLY for the 2 things we want the model to learn.
    instruction = """Analyze the following customer review and provide:
1. Whether it contains profanity (yes or no)
2. The sentiment (positive or neutral or negative)"""

    # ---------------------------------------------------------
    # 2. ATTACH THE REVIEW (Using your new 'Review' key)
    # ---------------------------------------------------------
    # Your JSON key is "Review", so we access row['Review']
    review_content = row['Review']

    # Combine them clearly
    user_content = f"{instruction}\n\nReview: {review_content}"

    # ---------------------------------------------------------
    # 3. CLEAN THE OUTPUT (Hybrid Strategy)
    # ---------------------------------------------------------
    # We strip out the "Rewritten:" line so the model acts as a pure classifier
    clean_output = "\n".join([
        line for line in row['output'].strip().split('\n')
        if line.startswith(("Profanity:", "Sentiment:"))
    ])

    # ---------------------------------------------------------
    # 4. FORMAT FOR LFM 2.5 (ChatML)
    # ---------------------------------------------------------
    text = f"""<|im_start|>system
You are a helpful AI assistant specialized in content moderation.<|im_end|>
<|im_start|>user
{user_content}<|im_end|>
<|im_start|>assistant
{clean_output}<|im_end|>"""

    return {"text": text}

# =========================================================
# HOW TO RUN IT
# =========================================================
dataset = Dataset.from_pandas(df)
dataset = dataset.map(format_lfm25_final)

# # Verify it looks right
print(dataset[0]['text'])

Map:   0%|          | 0/1364 [00:00<?, ? examples/s]

<|im_start|>system
You are a helpful AI assistant specialized in content moderation.<|im_end|>
<|im_start|>user
Analyze the following customer review and provide:
1. Whether it contains profanity (yes or no)
2. The sentiment (positive or neutral or negative)

Review:  crawls up the butt. hard to sleep in

Cute but no crotch snaps. Hard to get in and out of.<|im_end|>
<|im_start|>assistant
Profanity: yes
Sentiment: negative<|im_end|>


In [ ]:
# Assuming 'dataset' is your formatted Hugging Face dataset
# Split: 90% Training, 10% Validation (Test)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

# Print to verify
print(f"Training on: {len(dataset['train'])} examples")
print(f"Validating on: {len(dataset['test'])} examples")

Training on: 1227 examples
Validating on: 137 examples


In [ ]:
dataset['train'][5]

{'Review': ' My cats won’t touch them.\n\nCats are jerks. These two cats eat everything and they won’t even touch these stupid treats.',
 'output': 'Profanity: yes\nSentiment: negative\nRewritten: ',
 'text': '<|im_start|>system\nYou are a helpful AI assistant specialized in content moderation.<|im_end|>\n<|im_start|>user\nAnalyze the following customer review and provide:\n1. Whether it contains profanity (yes or no)\n2. The sentiment (positive or neutral or negative)\n\nReview:  My cats won’t touch them.\n\nCats are jerks. These two cats eat everything and they won’t even touch these stupid treats.<|im_end|>\n<|im_start|>assistant\nProfanity: yes\nSentiment: negative<|im_end|>'}

In [ ]:
training_args = SFTConfig(
    output_dir="/content/drive/MyDrive/lfm-profanity-lora",

    # CHANGED: 2 Epochs is safer for learning the format without overfitting
    num_train_epochs=5,

    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,  # Effective batch size = 16
    learning_rate=2e-4,

    logging_steps=5,      # Log loss frequently to watch for spikes

    # CRITICAL FIX: Lowered these so they actually run!
    # 85 steps per epoch -> evaluating every 20 steps gives you ~4 checks per epoch.
    save_steps=20,
    eval_steps=20,

    save_total_limit=2,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    eval_strategy="steps",
    load_best_model_at_end=True,
    warmup_steps=10,
    report_to="none",
    dataset_text_field="text",
    max_seq_length=2048,
)

In [ ]:
# ============================================================================
# STEP 6: Create Trainer
# ============================================================================

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],  # Add validation set
    tokenizer=tokenizer,
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1227 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/137 [00:00<?, ? examples/s]

In [ ]:
# ============================================================================
# STEP 7: Train!
# ============================================================================

print("\n🚀 Starting training...")
trainer_stats = trainer.train()

print("\n✅ Training complete!")
print(f"Training time: {trainer_stats.metrics['train_runtime']:.2f} seconds")

The model is already on multiple devices. Skipping the move to device specified in `args`.



🚀 Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,227 | Num Epochs = 5 | Total steps = 385
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 884,736 of 1,171,225,344 (0.08% trained)


Step,Training Loss,Validation Loss
20,1.302000,1.480073
40,1.298900,1.457605
60,1.268100,1.438580
80,1.377700,1.422920
100,1.283100,1.417726
120,1.231500,1.413287
140,1.227800,1.412130
160,1.310800,1.409106
180,1.321700,1.408277
200,1.244800,1.406221


Unsloth: Will smartly offload gradients to save VRAM!

✅ Training complete!
Training time: 922.66 seconds


In [ ]:
# ============================================================================
# STEP 8: Save Model
# ============================================================================

# Save LoRA adapters
model.save_pretrained("/content/drive/MyDrive/lfm-profanity-lora")
tokenizer.save_pretrained("/content/drive/MyDrive/lfm-profanity-lora")

print("\n✅ Model saved to Google Drive!")


✅ Model saved to Google Drive!


In [ ]:
def hybrid_analyze_review(review_text):
    # ====================================================
    # STEP 1: CLASSIFICATION (Adapter ON)
    # ====================================================
    detect_prompt = f"""<|im_start|>system
You are a helpful AI assistant specialized in content moderation.<|im_end|>
<|im_start|>user
Analyze the following customer review and provide:
1. Whether it contains profanity (yes/no)
2. The sentiment (positive/negative/neutral)

Review: {review_text}<|im_end|>
<|im_start|>assistant
"""
    inputs = tokenizer([detect_prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=64, temperature=0.1)
    analysis = tokenizer.decode(outputs[0], skip_special_tokens=True).split("assistant")[-1].strip()

    # Parse logic
    has_profanity = "Profanity: yes" in analysis
    is_positive = "Sentiment: positive" in analysis.lower()

    # ====================================================
    # STEP 2: REWRITE (Base Model - Adapter OFF)
    # ====================================================
    if has_profanity:

        if is_positive:
            # POSITIVE CASE
          system_task = (
              "You are a text rewriting engine. "
              "Your task is to remove profanity from enthusiastic praise while preserving excitement."
          )

          constraints = (
              "Transformation rules:\n"
              "- Preserve the original enthusiasm and positive emphasis.\n"
              "- Replace profanity with strong but professional praise.\n"
              "- Do NOT weaken the sentiment.\n"
              "- Do NOT summarize or neutralize excitement.\n"
              "- Output only the rewritten review text."
          )

          examples = """
          Original: Holy shit, this espresso machine is amazing!
          Rewritten: This espresso machine is absolutely amazing!

          Original: This game is fucking incredible. I can't put it down.
          Rewritten: This game is incredibly engaging. I cannot put it down.

          Original: Hell yeah! Arrived in one day. You guys are kickass.
          Rewritten: Excellent! It arrived in one day and the service was outstanding.
          """
          temperature = 0.6


        else:
            # NEGATIVE CASE
          system_task = (
              "You are a text rewriting engine. "
              "Your task is to remove profanity and insults while preserving the original complaint exactly."
          )

          constraints = (
              "Transformation rules:\n"
              "- Perform minimal rewriting. Replace only the profane or insulting words.\n"
              "- Preserve all concrete details such as actions, timelines, failures, and outcomes.\n"
              "- Do NOT summarize, generalize, or abstract the complaint.\n"
              "- Do NOT remove mentions of money, delays, damage, or lack of response.\n"
              "- Maintain a professional but firm tone.\n"
              "- Output only the rewritten review text."
          )

          examples = """
          Original: This laptop is a piece of shit. It broke after two days. What the hell?
          Rewritten: This laptop is unacceptable. It broke after two days.

          Original: Don't buy from this bastard seller. They took my money and ghosted me.
          Rewritten: Do not buy from this seller. They took my money and stopped responding.

          Original: The food tasted like crap and the waiter was a total ass.
          Rewritten: The food tasted poor and the waiter was unprofessional.
          """
          temperature = 0.3


        # -------- PROMPT ASSEMBLY --------
        rewrite_prompt = f"""<|im_start|>system
    {system_task}

    {constraints}
    <|im_end|>
    <|im_start|>user
    Below are examples of correct rewrites:

    {examples}

    Rewrite the following review.

    Original:
    {review_text}

    Rewritten:
    <|im_end|>
    <|im_start|>assistant
    """

        inputs = tokenizer([rewrite_prompt], return_tensors="pt").to("cuda")

        with model.disable_adapter():
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,
                temperature=temperature,
                do_sample=True,
            )

        rewrite = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # -------- SAFE CLEANUP --------
        rewrite = rewrite.split("assistant")[-1].strip()
        rewrite = rewrite.replace("Rewritten:", "").strip()
        rewrite = rewrite.splitlines()[0]

        return f"{analysis}\nRewritten: {rewrite}"

    else:
        return f"{analysis}\nRewritten: Not needed"

In [ ]:
def hybrid_analyze_review(review_text):
    # ====================================================
    # STEP 1: CLASSIFICATION (Adapter ON)
    # ====================================================
    detect_prompt = f"""<|im_start|>system
You are a helpful AI assistant specialized in content moderation.<|im_end|>
<|im_start|>user
Analyze the following customer review and provide:
1. Whether it contains profanity (yes/no)
2. The sentiment (positive/negative/neutral)

Review: {review_text}<|im_end|>
<|im_start|>assistant
"""
    inputs = tokenizer([detect_prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=64, temperature=0.1)
    analysis = (
        tokenizer.decode(outputs[0], skip_special_tokens=True)
        .split("assistant")[-1]
        .strip()
    )

    # Parse logic
    has_profanity = "Profanity: yes" in analysis
    is_positive = "Sentiment: positive" in analysis.lower()

    # ====================================================
    # STEP 2: REWRITE (Base Model - Adapter OFF)
    # ====================================================
    if has_profanity:
        if is_positive:
            # POSITIVE CASE
            system_task = (
                "You are a text rewriting engine. "
                "Your task is to remove profanity from enthusiastic praise while preserving excitement."
            )

            constraints = (
                "Transformation rules:\n"
                "- Preserve the original enthusiasm and positive emphasis.\n"
                "- Replace profanity with strong but professional praise.\n"
                "- Do NOT weaken the sentiment.\n"
                "- Do NOT summarize or neutralize excitement.\n"
                "- Output only the rewritten review text."
            )

            examples = """
          Original: Holy shit, this espresso machine is amazing!
          Rewritten: This espresso machine is absolutely amazing!

          Original: This game is fucking incredible. I can't put it down.
          Rewritten: This game is incredibly engaging. I cannot put it down.

          Original: Hell yeah! Arrived in one day. You guys are kickass.
          Rewritten: Excellent! It arrived in one day and the service was outstanding.
          """
            temperature = 0.6

        else:
            # NEGATIVE CASE
            system_task = (
                "You are a text rewriting engine. "
                "Your task is to remove profanity and insults while preserving the original complaint exactly."
            )

            constraints = (
                "Transformation rules:\n"
                "- Perform minimal rewriting. Replace only the profane or insulting words.\n"
                "- Preserve all concrete details such as actions, timelines, failures, and outcomes.\n"
                "- Do NOT summarize, generalize, or abstract the complaint.\n"
                "- Do NOT remove mentions of money, delays, damage, or lack of response.\n"
                "- Maintain a professional but firm tone.\n"
                "- Output only the rewritten review text."
            )

            examples = """
          Original: This laptop is a piece of shit. It broke after two days. What the hell?
          Rewritten: This laptop is unacceptable. It broke after two days.

          Original: Don't buy from this bastard seller. They took my money and ghosted me.
          Rewritten: Do not buy from this seller. They took my money and stopped responding.

          Original: The food tasted like crap and the waiter was a total ass.
          Rewritten: The food tasted poor and the waiter was unprofessional.
          """
            temperature = 0.3

        # -------- PROMPT ASSEMBLY --------
        rewrite_prompt = f"""<|im_start|>system
    {system_task}

    {constraints}
    <|im_end|>
    <|im_start|>user
    Below are examples of correct rewrites:

    {examples}

    Rewrite the following review.

    Original:
    {review_text}

    Rewritten:
    <|im_end|>
    <|im_start|>assistant
    """

        inputs = tokenizer([rewrite_prompt], return_tensors="pt").to("cuda")

        with model.disable_adapter():
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,
                temperature=temperature,
                do_sample=True,
            )

        rewrite = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # -------- SAFE CLEANUP --------
        rewrite = rewrite.split("assistant")[-1].strip()
        rewrite = rewrite.replace("Rewritten:", "").strip()
        rewrite = rewrite.splitlines()[0]

        return f"{analysis}\nRewritten: {rewrite}"

    else:
        return f"{analysis}\nRewritten: Not needed"


🚀 Running Hybrid Analysis on 12 examples...

📝 Review #1: This laptop is a piece of shit. It broke after two days. What the hell?
Profanity: yes
Sentiment: negative
Rewritten: This laptop is not satisfactory. It failed after just two days of use.
------------------------------------------------------------
📝 Review #2: Don't buy from this bastard seller. They took my money and ghosted me.
Profanity: yes
Sentiment: negative
Rewritten: Do not purchase from this seller. They delayed payment and did not respond to your inquiry.
------------------------------------------------------------
📝 Review #3: The food tasted like crap and the waiter was a total ass.
Profanity: yes
Sentiment: negative
Rewritten: The food quality was poor and the service was unsatisfactory.
------------------------------------------------------------
📝 Review #4: I am extremely disappointed. The item arrived damaged and the support team was unhelpful.
Profanity: no
Sentiment: negative
Rewritten: Not needed
----------

In [ ]:
stress_test_reviews = [
    # ==============================================================================
    # GROUP A: PROFANE & POSITIVE (The "Slang" Test)
    # Goal: Step 1 = Profanity: Yes | Step 2 = Rewrite (Keep enthusiasm, remove slang)
    # ==============================================================================
    "This graphics card is a beast! It runs Cyberpunk at 4k like a damn dream.",
    "Holy shit, the shipping was fast. Arrived in under 24 hours!",
    "I fucking love this band! Their sound quality is insane.",
    "Kickass customer support. They fixed my issue in ten minutes flat.",
    "Damn, these tacos are good. Best I've had in years.",
    "This vacuum sucks up dirt like a champ. Hell yeah!",
    "Badass design on this jacket. I get compliments everywhere.",
    "Unfuckingbelievable value for money. Highly recommend.",

    # ==============================================================================
    # GROUP B: PROFANE & NEGATIVE (The "Sanitization" Test)
    # Goal: Step 1 = Profanity: Yes | Step 2 = Rewrite (Keep complaint, remove insult)
    # ==============================================================================
    "This app is total garbage. It crashes every time I open it.",
    "The delivery driver threw my package on the porch. What an asshole.",
    "Stop calling me, you annoying bastards. I already cancelled.",
    "This piece of shit printer jammed on the very first page.",
    "The manager was a dick to his employees. I'm never coming back.",
    "Bullshit fees on my bill. This is robbery.",
    "Don't buy this crap. It broke after two days of light use.",
    "Customer service is a joke. They don't know what the hell they are doing.",

    # ==============================================================================
    # GROUP C: NON-PROFANE & NEGATIVE (The "False Positive" Test)
    # Goal: Step 1 = Profanity: No | Step 2 = Rewrite: Not Needed
    # (The model must NOT rewrite these even though they are angry)
    # ==============================================================================
    "I am very disappointed. The product does not match the description at all.",
    "Horrible experience. The food was cold and tasteless.",
    "It arrived damaged and the box was crushed. Very poor handling.",
    "Do not waste your money. The quality is extremely low for the price.",
    "The battery life is terrible. It dies after one hour.",
    "I waited 45 minutes on hold and nobody picked up. Frustrating.",
    "The color is completely wrong. It looks blue, not green.",
    "Unacceptable behavior from the staff. They were rude and dismissive.",

    # ==============================================================================
    # GROUP D: NON-PROFANE & POSITIVE (The "Baseline" Test)
    # Goal: Step 1 = Profanity: No | Step 2 = Rewrite: Not Needed
    # ==============================================================================
    "Absolutely wonderful experience! I will definitely return.",
    "Five stars. The build quality is solid and feels premium.",
    "A perfect gift for my mother. She was delighted.",
    "Super fast delivery and excellent packaging.",
    "The software is intuitive and very easy to use.",
    "Great value. It performs better than more expensive brands.",
    "I appreciate the quick response from the team. Thank you!",
    "Works exactly as advertised. No issues whatsoever."
]

# ====================================================
# RUN THE BATCH EXECUTION
# ====================================================
print(f"🚀 Running Stress Test on {len(stress_test_reviews)} reviews...\n")

for i, review in enumerate(stress_test_reviews):
    print(f"📝 Review #{i+1}: {review}")
    # Call your function
    result = hybrid_analyze_review(review)
    print(result)
    print("-" * 60)

🚀 Running Stress Test on 32 reviews...

📝 Review #1: This graphics card is a beast! It runs Cyberpunk at 4k like a damn dream.
Profanity: yes
Sentiment: positive
Rewritten: This graphics card performs exceptionally well, running Cyberpunk at 4k smoothly.
------------------------------------------------------------
📝 Review #2: Holy shit, the shipping was fast. Arrived in under 24 hours!
Profanity: yes
Sentiment: positive
Rewritten: The delivery arrived promptly and was received within the expected timeframe.
------------------------------------------------------------
📝 Review #3: I fucking love this band! Their sound quality is insane.
Profanity: yes
Sentiment: positive
Rewritten: I strongly recommend against purchasing from this band. Their performance was disappointing, and the sound quality fell short of expectations.
------------------------------------------------------------
📝 Review #4: Kickass customer support. They fixed my issue in ten minutes flat.
Profanity: no
Sentiment: 